# 10. Data Pipeline Automation
## 📚 Learning Objectives

By completing this notebook, you will:
- Build automated data processing pipelines
- Know how such pipelines are scheduled and orchestrated in production (concepts + pointers)
- Handle errors and retries in pipelines
- Track pipeline execution: duration, success/failure
- Automate repetitive data tasks

## 🔗 Prerequisites

- ✅ Example 05: Production Pipelines (understand pipeline structure)
- ✅ Example 06: Performance Optimization (optimize pipelines)
- ✅ Understanding of data processing workflows

---

## 📚 Prerequisites (What You Need First)

**BEFORE starting this notebook**, you should have completed:
- ✅ **Example 05: Production Pipelines** - Understand pipeline structure!
- ✅ **Example 06: Performance Optimization** - Optimize before automating!
- ✅ **Understanding of workflows**: What are data processing steps?

**If you haven't completed these**, you might struggle with:
- Understanding pipeline design
- Knowing what to automate
- Understanding scheduling and orchestration

---

## 🔗 Where This Notebook Fits

**This is the FINAL example in Unit 5: Extending the Scope of Data Science**

**Why pipeline automation?**
- **After** building pipelines, we automate them
- **Automation** saves time and reduces errors
- **Scheduling** ensures pipelines run regularly
- **Final step** in production ML systems

**Builds on**: 
- 📓 Example 05: Production Pipelines (pipeline structure)
- 📓 Example 06: Performance Optimization (optimized pipelines)

**Leads to**: 
- 📓 Production ML systems
- 📓 Automated data science workflows

**Why this order?**
1. Automation comes after building and optimizing
2. Essential for production systems
3. Final step in scaling data science

---

## The Story: From Manual to Automatic

Imagine you have a task you do daily (manual pipeline). **Before** automation, you do it manually every day. **After** automation, it runs automatically - saves time, reduces errors!

Same with data pipelines: **After** building pipelines, we automate - schedule runs, handle errors, monitor execution. **After** automation, pipelines run reliably!

---

## Why Pipeline Automation Matters

Pipeline automation is essential because:
- **Efficiency**: Saves time on repetitive tasks
- **Reliability**: Reduces human errors
- **Scalability**: Handles large workloads
- **Production**: Essential for production systems

**Common Student Questions:**
- **Q: What should I automate?**
  - Answer: Repetitive tasks, regular data processing, model retraining
  - Example: Daily data updates, weekly model retraining
  - Rule: If you do it regularly, automate it!
  
- **Q: How do I handle errors in automated pipelines?**
  - Answer: Use try/except, logging, retries, alerts
  - Example: Retry failed steps, log errors, send alerts
  - Benefit: Pipelines continue even when errors occur

---

## Introduction

**Data pipeline automation** transforms manual data processing into automated, scheduled workflows. This ensures reliable, efficient data processing in production environments.

**All concepts are explained in the code comments below - you can learn everything from this notebook alone!**

---

## 🔗 Automating Data Workflows

**Manual data processing is slow and error-prone!**
- Processing data manually takes too much time
- Errors happen when steps are forgotten
- We need automated, reliable pipelines

**This notebook teaches pipeline automation!**
- We'll learn **pipeline design** - structure data workflows
- We'll see **how scheduling works** - cron/Airflow concepts and pointers (the pipeline here runs once, by hand)
- We'll learn **error handling** - make pipelines robust
- We'll learn **monitoring** - track pipeline health

**This enables production-ready data pipelines!**

---

## Learning Objectives
1. Design automated data processing pipelines
2. Implement error handling and retries
3. Know how pipeline scheduling works in production (cron, Airflow - concepts + pointers)
4. Track pipeline runs: duration, success/failure

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Pipeline config
- scheduling/automation

**Outputs:** What you'll see when you run the cells

- Automated runs
- Logs

---

In [1]:
# WHAT: Import libraries and preview the automation topics.
# WHY: Pipelines that run themselves need explicit structure: steps, error handling, and run logs.

# Step 1: Import libraries
import pandas as pd
import numpy as np
from datetime import datetime
import time

print("✅ Libraries imported!")
print("\n📚 This notebook covers:")
print("   - Pipeline design")
print("   - Error handling")
print("   - Pipeline monitoring")
print("   - Automation strategies")

✅ Libraries imported!

📚 This notebook covers:
   - Pipeline design
   - Error handling
   - Pipeline monitoring
   - Automation strategies


## Step 2: Define Pipeline Steps

We define three functions: **extract** (get data), **transform** (process it), **load** (save it). Then we run them together with error handling.

In [2]:
# Extract: get data from source
def step1_extract():
    data = pd.DataFrame({'id': range(1, 101), 'value': np.random.rand(100) * 100})
    time.sleep(0.1)
    return data

# Transform: process and clean
def step2_transform(data):
    data = data.copy()
    data['value_normalized'] = (data['value'] - data['value'].mean()) / data['value'].std()
    time.sleep(0.1)
    return data

# Load: save to destination (simulated)
def step3_load(data):
    time.sleep(0.1)
    pass  # In production: write to DB, file, etc.

print("✅ Pipeline steps defined!")
print("   - Extract: Get data from source")
print("   - Transform: Process and clean data")
print("   - Load: Save processed data")

✅ Pipeline steps defined!
   - Extract: Get data from source
   - Transform: Process and clean data
   - Load: Save processed data


## Step 3: Execute Pipeline
Execute complete pipeline with error handling.


In [3]:
# WHAT: Chain extract -> transform -> load in run_pipeline() with try/except and a run log.
# WHY: The ETL wrapper turns loose scripts into a schedulable unit that reports success or failure.

print("\n" + "=" * 70)
print("=" * 70)

def run_pipeline():
    try:
        start_time = datetime.now()
        print(f"\n🚀 Starting pipeline at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        # Extract
        data = step1_extract()
        # Transform
        data = step2_transform(data)
        # Load
        step3_load(data)
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        print(f"\n✅ Pipeline completed successfully!")
        print(f"   Duration: {duration:.2f} seconds")
        return True
    except Exception as e:
        print(f"\n❌ Pipeline failed: {str(e)}")
        return False

# Run pipeline and record a small run log (the seed of monitoring)
run_started = datetime.now()
success = run_pipeline()
run_log = {
    'run_at': run_started.strftime('%Y-%m-%d %H:%M:%S'),
    'status': 'success' if success else 'failed',
    'duration_s': round((datetime.now() - run_started).total_seconds(), 2),
}
print(f"\n📒 Run log entry: {run_log}")
print("   In production this entry would be appended to a log store, and an")
print("   alert raised whenever status != success.")

print("""
💡 Scheduling & orchestration (how this runs WITHOUT a human):
   - cron / Task Scheduler: run `python pipeline.py` daily or hourly
   - Airflow / Prefect / Dagster: DAGs with retries, backfills, and alerting
   This notebook ran the pipeline once by hand; wiring it into one of the
   schedulers above is exactly how the same code becomes an automated job.""")

print("\n" + "=" * 70)
print("🎉 Example 10 complete - and with it, Course 05!")
print("   All 10 Unit 5 examples done. Finish with the Unit 5 exercise and quiz.")
print("=" * 70)



🚀 Starting pipeline at 2026-08-23 19:43:16



✅ Pipeline completed successfully!
   Duration: 0.31 seconds

📒 Run log entry: {'run_at': '2026-08-23 19:43:16', 'status': 'success', 'duration_s': 0.31}
   In production this entry would be appended to a log store, and an
   alert raised whenever status != success.

💡 Scheduling & orchestration (how this runs WITHOUT a human):
   - cron / Task Scheduler: run `python pipeline.py` daily or hourly
   - Airflow / Prefect / Dagster: DAGs with retries, backfills, and alerting
   This notebook ran the pipeline once by hand; wiring it into one of the
   schedulers above is exactly how the same code becomes an automated job.

🎉 Example 10 complete - and with it, Course 05!
   All 10 Unit 5 examples done. Finish with the Unit 5 exercise and quiz.


## 📚 References

1. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
2. Polyzotis, N., Roy, S., Whang, S. E., & Zinkevich, M. (2018). *Data Lifecycle Challenges in Production Machine Learning: A Survey*. ACM SIGMOD Record, 47(2), 17-28. <https://doi.org/10.1145/3299887.3299891>
3. Kreuzberger, D., Kühl, N., & Hirschl, S. (2022). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access, 11, 31866-31879. <https://arxiv.org/abs/2205.02302>